In [1]:
%pip install transformers bitsandbytes accelerate torch kernels

  Using cached transformers-5.6.2-py3-none-any.whl.metadata (33 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached kernels-0.13.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached huggingface_hub-1.12.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.4.4-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.24.2-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached tomlkit-0.14.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print(torch.cuda.memory_summary())

CUDA available: True
GPU name: NVIDIA H200 NVL
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from larg

In [3]:
from huggingface_hub import login
from getpass import getpass

hf_token = getpass("Paste your Hugging Face token: ")
login(token=hf_token)

Paste your Hugging Face token:  ········


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_name = "Qwen/Qwen3-Coder-30B-A3B-Instruct-FP8"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    torch_dtype="auto",
    device_map="auto",
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

Loading weights:   0%|          | 0/819 [00:00<?, ?it/s]

In [7]:
import os
import re
import ast
import json
import subprocess
import pandas as pd

BASE_DIR = os.getcwd()

def get_table_num(filename):
    match = re.search(r"LLM_statements_table_(\d+)\.txt$", filename)
    if match is None:
        return None
    return int(match.group(1))


def extract_statements_from_txt(raw: str) -> list[str]:
    """
    Extract statements from GPT-style outputs that contain a JSON object
    like {"statements": [...]}.
    """
    raw = raw.strip()

    match = re.search(r'(\{\s*"statements"\s*:\s*\[.*?\]\s*\})', raw, re.DOTALL)
    if not match:
        raise ValueError("Could not find a JSON object with a 'statements' field.")

    obj_text = match.group(1)

    try:
        obj = json.loads(obj_text)
    except Exception:
        obj = ast.literal_eval(obj_text)

    statements = obj.get("statements")
    if not isinstance(statements, list):
        raise ValueError("'statements' is not a list.")

    return [str(s).strip() for s in statements if str(s).strip()]


def extract_real_python(raw_code: str) -> str:
    """
    Keep only runnable Python from model output.
    Handles markdown fences, analysis/final tags, and extra prose.
    """
    text = raw_code.strip()

    # If the model used assistantfinal tags, keep only content after the final tag.
    final_tag_patterns = [
        r"</?assistantfinal>",
        r"assistantfinal",
    ]
    for pattern in final_tag_patterns:
        matches = list(re.finditer(pattern, text, flags=re.IGNORECASE))
        if matches:
            text = text[matches[-1].end():].strip()

    # Remove analysis blocks if present.
    text = re.sub(
        r"<analysis>.*?</analysis>",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    ).strip()

    # If markdown code fences exist, prefer the first python/plain fenced block.
    fence_match = re.search(
        r"```(?:python|py)?\s*(.*?)```",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    if fence_match:
        text = fence_match.group(1).strip()

    # If there is still prose before the script, start at the first import/from.
    code_start = re.search(r"(?m)^(import\s+|from\s+\S+\s+import\s+)", text)
    if code_start:
        text = text[code_start.start():].strip()

    return text


def get_generated_content(generation) -> str:
    """
    Extract assistant content from a transformers text-generation pipeline result.
    Works for chat-style and plain-text outputs.
    """
    generated_text = generation[0]["generated_text"]

    if isinstance(generated_text, list):
        return generated_text[-1].get("content", "")

    if isinstance(generated_text, str):
        return generated_text

    raise TypeError(f"Unexpected generated_text type: {type(generated_text)}")


def generate_code(source_model_name, b):
    inference_path = os.path.join(
        "..",
        "inference_generation",
        source_model_name,
        "b.LLM_Inferences",
    )

    if not os.path.isdir(inference_path):
        print(f"Missing inference folder: {inference_path}")
        return

    batch_start = b
    batch_end = b + 10

    statement_files = []

    for filename in os.listdir(inference_path):
        table_num = get_table_num(filename)

        if table_num is None:
            continue

        if batch_start <= table_num < batch_end:
            statement_files.append((table_num, filename))

    statement_files.sort()

    if not statement_files:
        print(f"No statement files found for {source_model_name}, batch {batch_start}-{batch_end - 1}.")
        return

    for table_num, statement_filename in statement_files:
        statement_path = os.path.join(inference_path, statement_filename)
        csv_path = os.path.join(
            "..",
            "inference_generation",
            "tables",
            f"table_{table_num}.csv",
        )

        if not os.path.exists(csv_path):
            print(f"No matching CSV found for {statement_filename}, skipping.")
            continue

        with open(statement_path, "r", encoding="utf-8") as f:
            raw_stmt_text = f.read()

        if not raw_stmt_text.strip():
            print(f"{statement_filename} is empty, skipping.")
            continue

        if source_model_name == "GPT":
            try:
                statements = extract_statements_from_txt(raw_stmt_text)
            except Exception as e:
                print(f"{statement_filename}: failed to parse statements -> {e}")
                continue
        else:
            statements = [
                line.strip()
                for line in raw_stmt_text.splitlines()
                if line.strip()
            ]

        if not statements:
            print(f"{statement_filename}: no statements parsed, skipping.")
            continue

        print(f"\nProcessing {statement_filename}.")
        print(f"Parsed {len(statements)} statements.")

        statements_text = "\n".join(
            f"{i + 1}. {stmt}" for i, stmt in enumerate(statements)
        )

        df = pd.read_csv(csv_path)
        sample_rows_text = df.head(3).to_csv(index=False)

        prompt2 = [
            {"role": "system", "content": "You are an expert data analyst."},
            {"role": "user", "content": f"""Do NOT repeat the instructions or the code provided.
Only output the requested Python code. Do NOT wrap the code in markdown fences.

Here's your task: Given the following statements:

{statements_text}

and the following CSV preview showing the header row and first 3 data rows:

{sample_rows_text}

write Python code using pandas that checks whether each statement is True or False and prints a justification.

The CSV is already located at: "{csv_path}". Hardcode this path directly in the script, with no sys.argv.

Requirements:
- Use pandas.
- Read the CSV from the hardcoded path.
- Convert any numeric columns stored as strings into numeric values when appropriate.
- Convert any turn numbers stored as strings into integers when appropriate.
- Check every statement listed above.
- Print whether each statement is True or False.
- Print a short justification for each result.
- Everything you output must be valid, immediately runnable Python.
- Do not include markdown, commentary, or explanations outside Python comments.

Here is an example structure to follow:

import pandas as pd

def print_result(statement_no: int, description: str, truth: bool, explanation: str):
    status = "TRUE" if truth else "FALSE"
    print(f"\\nStatement {{statement_no}}: {{status}}")
    print(f"  - {{description}}")
    print(f"  - Explanation: {{explanation}}")

def stmt_1(df: pd.DataFrame):
    \"\"\"1. For all individuals, if the person is a woman, then her age is between 21 and 43.\"\"\"
    women = df[df["gender"] == "F"]
    condition = women["age"].between(21, 43, inclusive="both")
    truth = condition.all()
    if truth:
        expl = f"All {{len(women)}} women are aged 21-43."
    else:
        viol = women[~condition]
        expl = f"{{len(viol)}} women violate the rule (ages: {{', '.join(map(str, viol['age'].tolist()))}})."
    return truth, expl

def main():
    df = pd.read_csv("{csv_path}")

    # Convert likely numeric columns safely.
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            pass

    checks = [(1, stmt_1)]  # extend for all statements

    for num, func in checks:
        truth, explanation = func(df)
        print_result(num, func.__doc__.strip(), truth, explanation)

if __name__ == "__main__":
    main()
"""}
        ]

        generation = generator(
            prompt2,
            do_sample=False,
            max_new_tokens=8000,
            eos_token_id=tokenizer.eos_token_id,
        )

        raw_code = get_generated_content(generation)
        python_code = extract_real_python(raw_code)

        if not python_code.strip():
            print(f"No Python code generated for table_{table_num}, skipping.")
            continue

        # Save generated Python files inside the notebook's code_generation directory.
        folder_path_python_code = os.path.join(
            BASE_DIR,
            source_model_name,
            "c.checking_statements",
        )
        os.makedirs(folder_path_python_code, exist_ok=True)

        python_file_name_LLM = f"python_code_table_{table_num}.py"
        full_path_py = os.path.join(folder_path_python_code, python_file_name_LLM)

        with open(full_path_py, "w", encoding="utf-8") as f:
            f.write(python_code)

        print(f"Saved {full_path_py}")

        # Execute generated Python file.
        print(f"Running {python_file_name_LLM}...")
        result = subprocess.run(
            ["python3", full_path_py],
            capture_output=True,
            text=True,
        )

        if result.stdout:
            print(result.stdout)

        if result.returncode != 0:
            print(f"[ERROR] Script exited with code {result.returncode}")
            print(result.stderr)
        else:
            print(f"[OK] {python_file_name_LLM} completed successfully.")

        # Save validation output inside the notebook's code_generation directory.
        folder_path_python_output_checking_statements = os.path.join(
            BASE_DIR,
            source_model_name,
            "d.checking_statements_output",
        )
        os.makedirs(folder_path_python_output_checking_statements, exist_ok=True)

        results_file_name = f"validation_inferences_table_{table_num}.txt"
        full_path_results_file = os.path.join(
            folder_path_python_output_checking_statements,
            results_file_name,
        )

        with open(full_path_results_file, "w", encoding="utf-8") as f:
            f.write(result.stdout)

            if result.returncode != 0:
                f.write(f"\n[ERROR] Script exited with code {result.returncode}\n")
                f.write(result.stderr)

        print(f"Saved {full_path_results_file}")


In [9]:
%%time

source_model_names = ["LLAMA"]
batches = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]

for source_model_name in source_model_names:
    for b in batches:
        generate_code(source_model_name, b)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'eos_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=8000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Processing LLM_statements_table_0.txt.
Parsed 16 statements.


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Fetching ... files: 0it [00:00, ?it/s]

Saved /home/ayushs13/code_generation/LLAMA/c.checking_statements/python_code_table_0.py
Running python_code_table_0.py...


...

Saved /home/ayushs13/code_generation/LLAMA/c.checking_statements/python_code_table_96.py
Running python_code_table_96.py...
[ERROR] Script exited with code 1
  File "/home/ayushs13/code_generation/LLAMA/c.checking_statements/python_code_table_96.py", line 645
    """55. If a
    ^
SyntaxError: unterminated triple-quoted string literal (detected at line 645)

Saved /home/ayushs13/code_generation/LLAMA/d.checking_statements_output/validation_inferences_table_96.txt

Processing LLM_statements_table_97.txt.
Parsed 18 statements.
Saved /home/ayushs13/code_generation/LLAMA/c.checking_statements/python_code_table_97.py
Running python_code_table_97.py...


[transformers] Both `max_new_tokens` (=8000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Statement 1: TRUE
  - 1. All sensors in the downtown zone have an average temperature greater than 21°C.
  - Explanation: All 5 downtown sensors have avg temp > 21°C.

Statement 2: TRUE
  - 2. All sensors in the industrial zone have an average humidity greater than 65%.
  - Explanation: All 2 industrial sensors have avg humidity > 65%.

Statement 3: TRUE
  - 3. If a sensor is in the residential zone, then its average noise level is greater than 64 dB.
  - Explanation: All 4 residential sensors have avg noise > 64 dB.

Statement 4: TRUE
  - 4. There exists at least one sensor in the park zone with an average PM2.5 level less than 20.
  - Explanation: At least one park sensor has PM2.5 < 20 (SN097009, 14.6).

Statement 5: FALSE
  - 5. All sensors with an average temperature less than 24°C have a foot traffic greater than 1000.
  - Explanation: 4 low-temp sensors violate (foot traffic: 865, 827, 680, 526).

Statement 6: TRUE
  - 6. If a sensor's average humidity is greater than 60%, then

[transformers] Both `max_new_tokens` (=8000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Statement 1: TRUE
  - 1. All farms with organic crops have a soil quality index greater than 68.
  - Explanation: All 9 organic farms have soil quality > 68.

Statement 2: TRUE
  - 2. If a farm grows soybean, then its yield is greater than 380 tons.
  - Explanation: All 4 soybean farms have yield > 380 tons.

Statement 3: TRUE
  - 3. There exists at least one farm with non-organic crops that has a soil quality index greater than 78.
  - Explanation: Found 1 non-organic farm(s) with soil quality > 78.

Statement 4: FALSE
  - 4. All farms with irrigation hours per week greater than 14 have a yield greater than 400 tons.
  - Explanation: 3 high irrigation farms violate the rule (yields: 360.5, 357.2, 300.5).

Statement 5: TRUE
  - 5. If a farm grows rice, then its acreage is less than 130.
  - Explanation: All 4 rice farms have acreage < 130.

Statement 6: TRUE
  - 6. Most farms with organic crops have a fertilizer usage greater than 700 kg.
  - Explanation: More than half (7/9) of organ